In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="09-similar-listing",
    notebook_name="02_embedding_techniques.ipynb"
)

# Embedding Techniques for Similar Listings

## The Big Idea (For a 12-Year-Old)

Imagine you have a giant box of 5 million LEGO houses. You want to organize them so that similar houses are close together on a shelf. But you can not look at the houses directly -- instead, you watch thousands of kids play. You notice that kids who pick up the red beach house also tend to pick up the blue beach house. Kids who pick up the mountain cabin tend to pick up the ski lodge.

By watching BEHAVIOR (which houses are picked up together), you learn which houses are similar -- without ever looking at the houses themselves!

This notebook dives deep into HOW we create these "similarity codes" (embeddings) for listings.

---

## Staff-Level Technical Summary

This notebook covers:
- **Word2Vec fundamentals**: CBOW vs Skip-gram, how context windows work
- **Adapting Word2Vec for sessions**: listing = word, session = sentence
- **Negative sampling**: why full softmax is infeasible, how NS approximates it
- **Hard negatives**: same-market negatives for fine-grained discrimination
- **Global context**: incorporating booking signal into the loss function
- **Feature-enriched embeddings**: incorporating listing attributes
- **Cold-start strategies**: handling new listings with no interaction data

## 1. Word2Vec Recap: The Foundation

Before we build Listing2Vec, we need to understand the original Word2Vec. It has two architectures:

### CBOW (Continuous Bag of Words)
- **Input**: surrounding context words
- **Output**: predict the center word
- Analogy: "Given the words around the blank, what word goes in the blank?"

### Skip-gram
- **Input**: the center word
- **Output**: predict each context word
- Analogy: "Given this word, what words are likely to surround it?"

**We use Skip-gram** because it works better with rare items (many listings have few interactions).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualize CBOW vs Skip-gram

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- CBOW ---
ax = axes[0]
words = ['the', 'cat', '___', 'on', 'the']
colors = ['#3498db', '#3498db', '#e74c3c', '#3498db', '#3498db']

for i, (w, c) in enumerate(zip(words, colors)):
    rect = plt.Rectangle((i * 2, 2), 1.5, 0.8, facecolor=c, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(i * 2 + 0.75, 2.4, w, ha='center', va='center', fontsize=12, fontweight='bold', color='white')

# Arrows from context to center
for i in [0, 1, 3, 4]:
    ax.annotate('', xy=(4.75, 2.0), xytext=(i * 2 + 0.75, 2.0),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))

# Prediction
pred_rect = plt.Rectangle((3.5, 0.3), 2.5, 0.8, facecolor='#f1c40f', edgecolor='black', linewidth=2)
ax.add_patch(pred_rect)
ax.text(4.75, 0.7, 'Predict: "sat"', ha='center', va='center', fontsize=11, fontweight='bold')
ax.annotate('', xy=(4.75, 1.1), xytext=(4.75, 1.95),
            arrowprops=dict(arrowstyle='->', color='#f39c12', lw=2))

ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.2, 3.5)
ax.set_title('CBOW\nContext words predict center word', fontsize=13, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

# --- Skip-gram ---
ax = axes[1]
words2 = ['___', '___', 'sat', '___', '___']
colors2 = ['#3498db', '#3498db', '#e74c3c', '#3498db', '#3498db']

for i, (w, c) in enumerate(zip(words2, colors2)):
    rect = plt.Rectangle((i * 2, 2), 1.5, 0.8, facecolor=c, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    display_text = w if w != '___' else '?'
    ax.text(i * 2 + 0.75, 2.4, display_text, ha='center', va='center', fontsize=12, fontweight='bold', color='white')

# Arrows from center to context
for i in [0, 1, 3, 4]:
    ax.annotate('', xy=(i * 2 + 0.75, 2.0), xytext=(4.75, 2.0),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))

# Predictions
preds = ['"the"', '"cat"', '"on"', '"the"']
pred_positions = [0, 1, 3, 4]
for p, pos in zip(preds, pred_positions):
    pred_rect = plt.Rectangle((pos * 2 - 0.25, 0.3), 2.0, 0.6, facecolor='#f1c40f', edgecolor='black', linewidth=1.5)
    ax.add_patch(pred_rect)
    ax.text(pos * 2 + 0.75, 0.6, p, ha='center', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.2, 3.5)
ax.set_title('Skip-gram (WE USE THIS)\nCenter word predicts each context word', fontsize=13, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

print("Why Skip-gram over CBOW for listings?")
print("  - Better for RARE items (many listings have few interactions)")
print("  - Skip-gram treats each (center, context) pair independently")
print("  - Works better with small datasets per item")

## 2. From Word2Vec to Listing2Vec

The key insight: **browsing sessions are like sentences, and listings are like words.**

| Word2Vec Concept | Listing2Vec Equivalent |
|-----------------|------------------------|
| Word | Listing ID |
| Sentence | User browsing session |
| Vocabulary | All listings on platform (~5M) |
| Context window | Nearby clicks in the session |
| "Words with similar meaning" | "Listings that users browse together" |

Just like Word2Vec learns that "king" and "queen" are similar because they appear in similar sentences, Listing2Vec learns that two beach houses are similar because they appear in the same browsing sessions.

In [ ]:
import random

random.seed(42)
np.random.seed(42)

# === Generate Synthetic Listing Data and Sessions ===

# Create 50 listings across 5 markets
markets = {
    'Miami Beach': {'ids': list(range(0, 10)), 'type': 'beach', 'price_range': (150, 400)},
    'NYC Manhattan': {'ids': list(range(10, 20)), 'type': 'city', 'price_range': (200, 600)},
    'Aspen Mountain': {'ids': list(range(20, 30)), 'type': 'mountain', 'price_range': (250, 500)},
    'LA Westside': {'ids': list(range(30, 40)), 'type': 'luxury', 'price_range': (300, 800)},
    'Austin Downtown': {'ids': list(range(40, 50)), 'type': 'urban', 'price_range': (100, 300)},
}

NUM_LISTINGS = 50

# Listing metadata
listing_market = {}  # listing_id -> market name
listing_price = {}   # listing_id -> price
for market_name, info in markets.items():
    for lid in info['ids']:
        listing_market[lid] = market_name
        listing_price[lid] = random.randint(*info['price_range'])

# Generate browsing sessions
# Key: sessions tend to contain listings from the SAME market
def generate_sessions(n_sessions=500):
    sessions = []
    for _ in range(n_sessions):
        # Pick a primary market (80% of clicks from here)
        primary_market = random.choice(list(markets.keys()))
        primary_ids = markets[primary_market]['ids']
        
        session_length = random.randint(3, 8)
        clicked = []
        for _ in range(session_length):
            if random.random() < 0.8:  # 80% from primary market
                clicked.append(random.choice(primary_ids))
            else:  # 20% from other markets (noise)
                clicked.append(random.randint(0, NUM_LISTINGS - 1))
        
        # Booked listing: always from primary market
        booked = random.choice(primary_ids)
        
        sessions.append({'clicked': clicked, 'booked': booked})
    
    return sessions

sessions = generate_sessions(500)

print(f"Generated {len(sessions)} sessions across {len(markets)} markets")
print(f"Total listings: {NUM_LISTINGS}")
print(f"\nSample sessions:")
for i in range(5):
    s = sessions[i]
    market_of_booked = listing_market[s['booked']]
    print(f"  Session {i+1}: {s['clicked']} -> booked {s['booked']} ({market_of_booked})")

print(f"\nMarkets:")
for name, info in markets.items():
    print(f"  {name}: listings {info['ids'][0]}-{info['ids'][-1]}, type={info['type']}, price={info['price_range']}")

## 3. Implementing Word2Vec from Scratch

Before adapting it for listings, let us build a basic Word2Vec with negative sampling from scratch. This helps us understand every component.

### The Skip-gram Objective

For a center listing `c` and context listing `p`:

```
P(p is context of c) = sigmoid(E_c . E_p)
```

We want this probability to be HIGH for real context pairs, and LOW for random pairs.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class Word2VecSkipGram(nn.Module):
    """
    Word2Vec Skip-gram with negative sampling, built from scratch.
    
    12-year-old version:
    Every item (word or listing) gets a secret code -- a list of numbers.
    We train the model so that items appearing together have SIMILAR codes
    (their numbers are close), and random items have DIFFERENT codes.
    
    Staff-level detail:
    - Two embedding matrices: center embeddings (W) and context embeddings (W')
    - In practice, we only use the center embeddings after training
    - Negative sampling replaces the expensive softmax over full vocabulary
    - Loss = -log(sig(w_c . w_p)) - SUM_neg log(sig(-w_c . w_n))
    """
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Center embeddings (the ones we keep after training)
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Context embeddings (used during training only)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize with small values
        nn.init.xavier_uniform_(self.center_embeddings.weight)
        nn.init.xavier_uniform_(self.context_embeddings.weight)
    
    def forward(self, center_ids, context_ids, negative_ids):
        """
        Compute the negative sampling loss.
        
        center_ids:   (batch_size,)
        context_ids:  (batch_size,)
        negative_ids: (batch_size, num_negatives)
        """
        # Get embeddings
        center_emb = self.center_embeddings(center_ids)        # (batch, dim)
        context_emb = self.context_embeddings(context_ids)      # (batch, dim)
        negative_emb = self.context_embeddings(negative_ids)    # (batch, num_neg, dim)
        
        # Positive score: dot product of center and context
        pos_score = (center_emb * context_emb).sum(dim=1)       # (batch,)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10)  # (batch,)
        
        # Negative scores: dot product of center and each negative
        # center_emb: (batch, dim) -> (batch, 1, dim) for broadcasting
        neg_score = torch.bmm(negative_emb, center_emb.unsqueeze(2)).squeeze(2)  # (batch, num_neg)
        neg_loss = -torch.log(torch.sigmoid(-neg_score) + 1e-10)  # (batch, num_neg)
        neg_loss = neg_loss.sum(dim=1)  # (batch,)
        
        # Total loss
        total_loss = (pos_loss + neg_loss).mean()
        return total_loss
    
    def get_embeddings(self):
        """Return the center embeddings (the learned representations)."""
        return self.center_embeddings.weight.detach().numpy()


print("Word2Vec Skip-gram architecture:")
print("  Input:  center listing ID")
print("  Output: probability of each listing being in the context")
print("")
print("Loss function (negative sampling):")
print("  L = -log(sig(E_center . E_positive)) - SUM_neg log(sig(-E_center . E_negative))")
print("")
print("The first term PUSHES center toward positive (context) listings.")
print("The second term PUSHES center away from negative (random) listings.")

In [ ]:
# === Prepare Training Data ===

def generate_skipgram_pairs(sessions, window_size=2, num_negatives=5, vocab_size=50):
    """
    Generate (center, positive_context, negative_samples) triples from sessions.
    
    12-year-old version:
    For each listing in a session, we pair it with its neighbors (positive)
    and with random listings (negative). The model learns to tell them apart.
    """
    centers = []
    positives = []
    negatives = []
    
    for session in sessions:
        clicked = session['clicked']
        
        for center_idx in range(len(clicked)):
            center = clicked[center_idx]
            
            # Get context listings within window
            start = max(0, center_idx - window_size)
            end = min(len(clicked), center_idx + window_size + 1)
            context_set = set(clicked[start:end])
            context_set.discard(center)  # Remove self
            
            for ctx in context_set:
                # Sample negatives (random listings not in context)
                neg_samples = []
                while len(neg_samples) < num_negatives:
                    neg = random.randint(0, vocab_size - 1)
                    if neg != center and neg not in context_set:
                        neg_samples.append(neg)
                
                centers.append(center)
                positives.append(ctx)
                negatives.append(neg_samples)
    
    return (
        torch.tensor(centers),
        torch.tensor(positives),
        torch.tensor(negatives)
    )


WINDOW_SIZE = 2
NUM_NEGATIVES = 5
EMBEDDING_DIM = 32

center_ids, positive_ids, negative_ids = generate_skipgram_pairs(
    sessions, window_size=WINDOW_SIZE, num_negatives=NUM_NEGATIVES, vocab_size=NUM_LISTINGS
)

print(f"Training data generated:")
print(f"  Total training pairs: {len(center_ids)}")
print(f"  Window size: {WINDOW_SIZE}")
print(f"  Negatives per pair: {NUM_NEGATIVES}")
print(f"\nSample training triple:")
print(f"  Center: listing {center_ids[0].item()} ({listing_market[center_ids[0].item()]})")
print(f"  Positive: listing {positive_ids[0].item()} ({listing_market[positive_ids[0].item()]})")
negs = negative_ids[0].tolist()
print(f"  Negatives: {negs} ({[listing_market[n] for n in negs]})")

In [ ]:
# === Train the Basic Word2Vec Model ===

model = Word2VecSkipGram(NUM_LISTINGS, EMBEDDING_DIM)
optimizer = optim.Adam(model.parameters(), lr=0.005)

# Mini-batch training
batch_size = 256
n_samples = len(center_ids)
n_epochs = 50

print(f"=== Training Basic Word2Vec (Skip-gram + Negative Sampling) ===")
print(f"Vocab size: {NUM_LISTINGS}, Embedding dim: {EMBEDDING_DIM}")
print(f"Batch size: {batch_size}, Epochs: {n_epochs}")
print()

losses = []
for epoch in range(n_epochs):
    # Shuffle data
    perm = torch.randperm(n_samples)
    epoch_loss = 0
    n_batches = 0
    
    for i in range(0, n_samples, batch_size):
        idx = perm[i:i+batch_size]
        
        batch_centers = center_ids[idx]
        batch_positives = positive_ids[idx]
        batch_negatives = negative_ids[idx]
        
        optimizer.zero_grad()
        loss = model(batch_centers, batch_positives, batch_negatives)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")

# Plot loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#3498db', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Basic Word2Vec Training Loss', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# === Visualize Basic Embeddings ===

embeddings = model.get_embeddings()

# PCA to 2D
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

market_colors = {
    'Miami Beach': '#e74c3c',
    'NYC Manhattan': '#3498db',
    'Aspen Mountain': '#2ecc71',
    'LA Westside': '#9b59b6',
    'Austin Downtown': '#f39c12'
}

fig, ax = plt.subplots(figsize=(10, 8))

for market_name, info in markets.items():
    ids = info['ids']
    color = market_colors[market_name]
    ax.scatter(emb_2d[ids, 0], emb_2d[ids, 1], c=color, s=100, 
               label=market_name, edgecolors='black', linewidth=0.5, alpha=0.8)

ax.legend(fontsize=11, loc='best')
ax.set_xlabel('PCA Dimension 1', fontsize=12)
ax.set_ylabel('PCA Dimension 2', fontsize=12)
ax.set_title('Basic Word2Vec Embeddings\nListings from same market cluster together', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The model learned market structure purely from co-occurrence patterns!")
print("No listing attributes (city, price, type) were used during training.")

## 4. Why Negative Sampling Matters

### The Problem with Full Softmax

**Simple explanation**: In the original Word2Vec, to predict the probability of one context word, you need to compute a score for EVERY word in the vocabulary (5 million listings!). That is like checking every single book in the library to figure out which one goes next to Harry Potter.

**Technical explanation**: The full softmax requires computing:

```
P(context | center) = exp(E_context . E_center) / SUM_ALL_LISTINGS exp(E_j . E_center)
```

That denominator sums over ALL 5 million listings -- way too expensive for every training step.

### The Solution: Negative Sampling

Instead of computing probability over all listings, we only compute it for:
- The actual positive context listing (should be high)
- A few randomly sampled negatives (should be low)

This reduces computation from O(V) to O(k), where k = number of negatives (typically 5-20).

In [ ]:
# Visualize why negative sampling is efficient

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full Softmax
ax = axes[0]
V = 50  # vocab size for visualization
bars = ax.bar(range(V), np.ones(V), color='#e74c3c', alpha=0.3, edgecolor='#c0392b')
bars[25].set_color('#2ecc71')  # The one positive
bars[25].set_alpha(1.0)
ax.annotate('Positive\n(1 listing)', xy=(25, 1.05), ha='center', fontsize=10, fontweight='bold', color='#27ae60')
ax.set_title(f'Full Softmax\nCompute score for ALL {V} listings', fontsize=12, fontweight='bold')
ax.set_xlabel('Listing index', fontsize=10)
ax.set_ylabel('Must compute', fontsize=10)
ax.text(25, 0.5, f'O({V}) per\ntraining step', ha='center', fontsize=11, fontweight='bold', color='#c0392b')

# Negative Sampling
ax = axes[1]
k = 5
selected = [25] + random.sample([i for i in range(V) if i != 25], k)
bars2 = ax.bar(range(V), np.ones(V), color='#ecf0f1', alpha=0.3, edgecolor='#bdc3c7')
for s in selected:
    bars2[s].set_color('#e74c3c' if s != 25 else '#2ecc71')
    bars2[s].set_alpha(1.0)
ax.annotate('Positive\n(1 listing)', xy=(25, 1.05), ha='center', fontsize=10, fontweight='bold', color='#27ae60')
ax.set_title(f'Negative Sampling\nCompute score for 1 + {k} = {k+1} listings only', fontsize=12, fontweight='bold')
ax.set_xlabel('Listing index', fontsize=10)
ax.set_ylabel('Must compute', fontsize=10)
ax.text(25, 0.5, f'O({k+1}) per\ntraining step', ha='center', fontsize=11, fontweight='bold', color='#27ae60')

plt.tight_layout()
plt.show()

print(f"Speedup: from computing {V} scores to {k+1} scores per training step")
print(f"In production (5M listings): from 5,000,000 to {k+1} computations!")
print(f"That is a {5_000_000 / (k+1):.0f}x speedup.")

## 5. Hard Negatives: The Secret Sauce

### The Problem with Random Negatives

**Simple explanation**: If you randomly pick a "not similar" listing, it is probably in a completely different city. That is too easy -- of course a Miami beach house is different from an Aspen ski cabin! The hard question is: among all the beach houses in Miami, which ones are similar and which are NOT?

**Technical explanation**: Random negatives come from the full listing pool, so they are overwhelmingly from different geographic markets. The model easily learns cross-market distinctions but fails at intra-market discrimination.

In [ ]:
# === Demonstrate the difference between random vs hard negatives ===

def analyze_negative_distribution(sessions, vocab_size, num_trials=1000):
    """
    Show that random negatives are mostly from DIFFERENT markets (easy)
    while hard negatives are from the SAME market (hard).
    """
    same_market_count = 0
    diff_market_count = 0
    
    for _ in range(num_trials):
        # Pick a random center listing
        center = random.randint(0, vocab_size - 1)
        center_market = listing_market[center]
        
        # Pick a random negative
        neg = random.randint(0, vocab_size - 1)
        neg_market = listing_market[neg]
        
        if neg_market == center_market:
            same_market_count += 1
        else:
            diff_market_count += 1
    
    return same_market_count, diff_market_count


same, diff = analyze_negative_distribution(sessions, NUM_LISTINGS)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Random negatives distribution
ax = axes[0]
ax.bar(['Same Market\n(Hard)', 'Different Market\n(Easy)'], [same, diff],
       color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=1)
ax.set_title('Random Negative Sampling', fontsize=13, fontweight='bold')
ax.set_ylabel('Count', fontsize=11)
ax.text(0, same + 20, f'{same/10:.0f}%', ha='center', fontsize=12, fontweight='bold')
ax.text(1, diff + 20, f'{diff/10:.0f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, max(same, diff) * 1.15)

# Hard negatives (forced same market)
ax = axes[1]
ax.bar(['Same Market\n(Hard)', 'Different Market\n(Easy)'], [1000, 0],
       color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=1)
ax.set_title('Hard Negative Sampling', fontsize=13, fontweight='bold')
ax.set_ylabel('Count', fontsize=11)
ax.text(0, 1020, '100%', ha='center', fontsize=12, fontweight='bold')
ax.text(1, 20, '0%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1150)

plt.suptitle('Why Hard Negatives Matter', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"With random sampling, only {same/10:.0f}% of negatives are from the same market.")
print(f"The model learns to distinguish Miami vs NYC (easy) but NOT")
print(f"Miami Beach House A vs Miami Beach House B (the hard, important part).")
print(f"\nHard negatives: ALWAYS from the same market -> forces fine-grained learning.")

In [ ]:
# === Listing2Vec with Hard Negatives and Global Context ===

class Listing2Vec(nn.Module):
    """
    Full Listing2Vec model with two improvements over basic Word2Vec:
    1. Global context (booked listing)
    2. Hard negatives (same-market negatives)
    
    12-year-old version:
    Same as before, but with two super-powers:
    - Power 1: The house someone ACTUALLY BOOKED gets extra attention.
               It stays as a positive partner for EVERY listing in the session.
    - Power 2: Instead of comparing against random houses from any city,
               we compare against houses in the SAME neighborhood.
               This is like a harder test that makes the model smarter.
    
    Staff-level detail:
    Loss = -SUM(D_pos)  log(sig(E_c . E_p))
           -SUM(D_neg)  log(sig(-E_c . E_n))
           -SUM(D_book) log(sig(E_c . E_book))      # Global context
           -SUM(D_hard) log(sig(-E_c . E_hard))      # Hard negatives
    """
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        nn.init.xavier_uniform_(self.center_embeddings.weight)
        nn.init.xavier_uniform_(self.context_embeddings.weight)
    
    def forward(self, center_ids, positive_ids, negative_ids, 
                booked_ids=None, hard_negative_ids=None):
        """
        Compute the improved loss with global context and hard negatives.
        
        center_ids:        (batch,)
        positive_ids:      (batch,)
        negative_ids:      (batch, num_neg)
        booked_ids:        (batch,) -- the eventually booked listing (global context)
        hard_negative_ids: (batch, num_hard_neg) -- same-market negatives
        """
        center_emb = self.center_embeddings(center_ids)     # (batch, dim)
        context_emb = self.context_embeddings(positive_ids)  # (batch, dim)
        neg_emb = self.context_embeddings(negative_ids)      # (batch, num_neg, dim)
        
        # Component 1: Positive pairs (push center toward context)
        pos_score = (center_emb * context_emb).sum(dim=1)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10)
        
        # Component 2: Random negatives (push center away from random)
        neg_score = torch.bmm(neg_emb, center_emb.unsqueeze(2)).squeeze(2)
        neg_loss = -torch.log(torch.sigmoid(-neg_score) + 1e-10).sum(dim=1)
        
        total_loss = pos_loss + neg_loss
        
        # Component 3: Global context -- booked listing (push center toward booked)
        if booked_ids is not None:
            booked_emb = self.context_embeddings(booked_ids)
            book_score = (center_emb * booked_emb).sum(dim=1)
            book_loss = -torch.log(torch.sigmoid(book_score) + 1e-10)
            total_loss = total_loss + book_loss
        
        # Component 4: Hard negatives (push center away from same-market non-context)
        if hard_negative_ids is not None:
            hard_neg_emb = self.context_embeddings(hard_negative_ids)
            hard_neg_score = torch.bmm(hard_neg_emb, center_emb.unsqueeze(2)).squeeze(2)
            hard_neg_loss = -torch.log(torch.sigmoid(-hard_neg_score) + 1e-10).sum(dim=1)
            total_loss = total_loss + hard_neg_loss
        
        return total_loss.mean()
    
    def get_embeddings(self):
        return self.center_embeddings.weight.detach().numpy()


print("Listing2Vec model created with 4 loss components:")
print("  1. Positive pairs:      Push center TOWARD clicked neighbors")
print("  2. Random negatives:    Push center AWAY from random listings")
print("  3. Global context:      Push center TOWARD booked listing")
print("  4. Hard negatives:      Push center AWAY from same-market non-context")

In [ ]:
# === Prepare Training Data with Global Context and Hard Negatives ===

def generate_improved_training_data(sessions, window_size=2, num_negatives=5, 
                                     num_hard_negatives=3, vocab_size=50):
    """
    Generate training data with all 4 components.
    
    Key additions vs basic Word2Vec:
    - booked_ids: the eventually booked listing for each session
    - hard_negative_ids: same-market listings NOT in the context window
    """
    all_centers = []
    all_positives = []
    all_negatives = []
    all_booked = []
    all_hard_negatives = []
    
    for session in sessions:
        clicked = session['clicked']
        booked = session['booked']
        booked_market = listing_market[booked]
        
        for center_idx in range(len(clicked)):
            center = clicked[center_idx]
            center_market = listing_market[center]
            
            # Context window
            start = max(0, center_idx - window_size)
            end = min(len(clicked), center_idx + window_size + 1)
            context_set = set(clicked[start:end])
            context_set.discard(center)
            
            for ctx in context_set:
                # Random negatives
                neg_samples = []
                while len(neg_samples) < num_negatives:
                    neg = random.randint(0, vocab_size - 1)
                    if neg != center and neg not in context_set:
                        neg_samples.append(neg)
                
                # Hard negatives: same market as center, NOT in context
                same_market_ids = markets[center_market]['ids']
                hard_neg_candidates = [l for l in same_market_ids 
                                       if l != center and l not in context_set]
                hard_neg_samples = []
                for _ in range(num_hard_negatives):
                    if hard_neg_candidates:
                        hard_neg_samples.append(random.choice(hard_neg_candidates))
                    else:
                        hard_neg_samples.append(random.randint(0, vocab_size - 1))
                
                all_centers.append(center)
                all_positives.append(ctx)
                all_negatives.append(neg_samples)
                all_booked.append(booked)
                all_hard_negatives.append(hard_neg_samples)
    
    return (
        torch.tensor(all_centers),
        torch.tensor(all_positives),
        torch.tensor(all_negatives),
        torch.tensor(all_booked),
        torch.tensor(all_hard_negatives)
    )


centers, positives, negatives, booked, hard_negs = generate_improved_training_data(
    sessions, window_size=2, num_negatives=5, num_hard_negatives=3, vocab_size=NUM_LISTINGS
)

print(f"Improved training data:")
print(f"  Total pairs: {len(centers)}")
print(f"  Each pair has: center, positive, {negatives.shape[1]} random negs, booked, {hard_negs.shape[1]} hard negs")
print(f"\nSample:")
print(f"  Center:         listing {centers[0].item()} ({listing_market[centers[0].item()]})")
print(f"  Positive:       listing {positives[0].item()} ({listing_market[positives[0].item()]})")
print(f"  Random negs:    {negatives[0].tolist()}")
print(f"  Booked (global): listing {booked[0].item()} ({listing_market[booked[0].item()]})")
print(f"  Hard negs:      {hard_negs[0].tolist()} (same market as center!)")

In [ ]:
# === Train the Improved Listing2Vec ===

improved_model = Listing2Vec(NUM_LISTINGS, EMBEDDING_DIM)
optimizer = optim.Adam(improved_model.parameters(), lr=0.005)

batch_size = 256
n_samples = len(centers)
n_epochs = 50

print(f"=== Training Improved Listing2Vec ===")
print(f"With global context (booked) + hard negatives (same market)")
print()

improved_losses = []
for epoch in range(n_epochs):
    perm = torch.randperm(n_samples)
    epoch_loss = 0
    n_batches = 0
    
    for i in range(0, n_samples, batch_size):
        idx = perm[i:i+batch_size]
        
        optimizer.zero_grad()
        loss = improved_model(
            centers[idx], positives[idx], negatives[idx],
            booked_ids=booked[idx],
            hard_negative_ids=hard_negs[idx]
        )
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    improved_losses.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}")

# Compare loss curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#3498db', linewidth=2, label='Basic Word2Vec')
ax.plot(improved_losses, color='#e74c3c', linewidth=2, label='Improved Listing2Vec')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss: Basic vs Improved', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nNote: The improved model has higher loss because it is solving a HARDER problem.")
print("Hard negatives force the model to learn finer distinctions within the same market.")

In [ ]:
# === Compare embedding quality: basic vs improved ===

basic_embeddings = model.get_embeddings()
improved_embeddings = improved_model.get_embeddings()

def compute_intra_vs_inter_similarity(embeddings, markets_dict, listing_market_map):
    """
    Compute average similarity WITHIN markets vs ACROSS markets.
    Good embeddings should have high intra-market similarity and low inter-market similarity.
    """
    sim_matrix = cosine_similarity(embeddings)
    n = len(embeddings)
    
    intra_sims = []
    inter_sims = []
    
    for i in range(n):
        for j in range(i + 1, n):
            sim = sim_matrix[i, j]
            if listing_market_map[i] == listing_market_map[j]:
                intra_sims.append(sim)
            else:
                inter_sims.append(sim)
    
    return np.mean(intra_sims), np.mean(inter_sims)


basic_intra, basic_inter = compute_intra_vs_inter_similarity(basic_embeddings, markets, listing_market)
improved_intra, improved_inter = compute_intra_vs_inter_similarity(improved_embeddings, markets, listing_market)

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(2)
width = 0.35

bars1 = ax.bar(x - width/2, [basic_intra, basic_inter], width, 
               label='Basic Word2Vec', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, [improved_intra, improved_inter], width, 
               label='Improved Listing2Vec', color='#e74c3c', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(['Intra-Market Similarity\n(same city -- should be HIGH)', 
                     'Inter-Market Similarity\n(different city -- should be LOW)'], fontsize=10)
ax.set_ylabel('Average Cosine Similarity', fontsize=11)
ax.set_title('Embedding Quality: Intra-Market vs Inter-Market Similarity', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

gap_basic = basic_intra - basic_inter
gap_improved = improved_intra - improved_inter
print(f"Discrimination gap (intra - inter):")
print(f"  Basic Word2Vec:        {gap_basic:.4f}")
print(f"  Improved Listing2Vec:  {gap_improved:.4f}")
print(f"\nHigher gap = better discrimination between markets.")

In [ ]:
# === Visualize Improved Embeddings Side-by-Side ===

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, embs, title in [(axes[0], basic_embeddings, 'Basic Word2Vec'),
                         (axes[1], improved_embeddings, 'Improved Listing2Vec')]:
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embs)
    
    for market_name, info in markets.items():
        ids = info['ids']
        color = market_colors[market_name]
        ax.scatter(emb_2d[ids, 0], emb_2d[ids, 1], c=color, s=80,
                   label=market_name, edgecolors='black', linewidth=0.5, alpha=0.8)
    
    ax.legend(fontsize=9, loc='best')
    ax.set_xlabel('PCA Dim 1', fontsize=11)
    ax.set_ylabel('PCA Dim 2', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Embedding Space Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Cold-Start Handling for New Listings

### The Problem

**Simple explanation**: When a brand new listing is added to the platform, nobody has clicked on it yet. It has no browsing history, so the model has no embedding for it. How do we recommend it (or recommend things similar to it)?

### Strategies

1. **Geographic Proxy**: Use the embedding of the nearest known listing in the same neighborhood. If a new beach house appears in South Beach, use the average embedding of existing South Beach listings.

2. **Feature-Based Initialization**: Use listing attributes (price, type, location) to initialize the embedding based on similar listings' embeddings.

3. **Daily Retraining**: After 1 day, the listing accumulates enough interaction data to be included in the next daily training cycle.

In [ ]:
# === Cold-Start Strategies ===

def cold_start_geographic_proxy(new_listing_market, embeddings, markets_dict):
    """
    Strategy 1: Use average embedding of same-market listings.
    
    12-year-old version:
    A brand new beach house in Miami has no embedding. So we take the
    AVERAGE of all existing Miami beach house embeddings and use that
    as a starting point. It is not perfect, but it is a reasonable guess.
    """
    same_market_ids = markets_dict[new_listing_market]['ids']
    same_market_embeddings = embeddings[same_market_ids]
    return np.mean(same_market_embeddings, axis=0)


def cold_start_nearest_listing(new_price, new_market, embeddings, markets_dict):
    """
    Strategy 2: Use embedding of the listing most similar in features.
    Match on price and market.
    """
    same_market_ids = markets_dict[new_market]['ids']
    best_id = None
    best_diff = float('inf')
    
    for lid in same_market_ids:
        price_diff = abs(listing_price[lid] - new_price)
        if price_diff < best_diff:
            best_diff = price_diff
            best_id = lid
    
    return embeddings[best_id], best_id


# Demonstrate cold start
new_listing_market = 'Miami Beach'
new_listing_price = 220

proxy_emb = cold_start_geographic_proxy(new_listing_market, improved_embeddings, markets)
nearest_emb, nearest_id = cold_start_nearest_listing(new_listing_price, new_listing_market, 
                                                      improved_embeddings, markets)

# Check quality: which existing listings are most similar to each cold-start embedding?
proxy_sims = cosine_similarity([proxy_emb], improved_embeddings)[0]
nearest_sims = cosine_similarity([nearest_emb], improved_embeddings)[0]

print("=== Cold-Start Strategies for a New Miami Beach House ($220/night) ===")

print("\nStrategy 1: Geographic Proxy (average of Miami Beach embeddings)")
top_5_proxy = np.argsort(-proxy_sims)[:5]
for i, idx in enumerate(top_5_proxy):
    print(f"  #{i+1}: Listing {idx} ({listing_market[idx]}, ${listing_price[idx]}) sim={proxy_sims[idx]:.3f}")

print(f"\nStrategy 2: Nearest Listing (closest price in same market)")
print(f"  Using listing {nearest_id} (${listing_price[nearest_id]}/night) as proxy")
top_5_nearest = np.argsort(-nearest_sims)[:5]
for i, idx in enumerate(top_5_nearest):
    print(f"  #{i+1}: Listing {idx} ({listing_market[idx]}, ${listing_price[idx]}) sim={nearest_sims[idx]:.3f}")

print("\nBoth strategies produce Miami Beach listings as most similar -- cold start works!")
print("After 1 day of user interactions, daily retraining gives the new listing its own embedding.")

In [ ]:
# Visualize cold start embedding in the space

fig, ax = plt.subplots(figsize=(10, 8))

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(improved_embeddings)

# Plot existing listings
for market_name, info in markets.items():
    ids = info['ids']
    color = market_colors[market_name]
    ax.scatter(emb_2d[ids, 0], emb_2d[ids, 1], c=color, s=80,
               label=market_name, edgecolors='black', linewidth=0.5, alpha=0.7)

# Plot cold-start embeddings
proxy_2d = pca.transform([proxy_emb])[0]
nearest_2d = pca.transform([nearest_emb])[0]

ax.scatter([proxy_2d[0]], [proxy_2d[1]], c='gold', s=300, marker='*', 
           edgecolors='black', linewidth=2, zorder=5, label='Cold Start: Geographic Proxy')
ax.scatter([nearest_2d[0]], [nearest_2d[1]], c='cyan', s=300, marker='D', 
           edgecolors='black', linewidth=2, zorder=5, label='Cold Start: Nearest Listing')

ax.legend(fontsize=9, loc='best')
ax.set_xlabel('PCA Dim 1', fontsize=12)
ax.set_ylabel('PCA Dim 2', fontsize=12)
ax.set_title('Cold-Start Embeddings Land in the Correct Region', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Incorporating Listing Features into Embeddings

### Beyond Pure Behavioral Data

**Simple explanation**: So far, the model only learns from browsing behavior. But what if we also tell it about listing features -- like price, type, and location? This gives the model extra clues, especially for new listings with little browsing data.

**Technical explanation**: We can concatenate listing feature embeddings with the learned co-occurrence embeddings. For example:

```
Final embedding = [behavioral_embedding ; price_bucket_embedding ; type_embedding ; city_embedding]
```

This is useful for cold-start listings and adds interpretability.

In [ ]:
class FeatureEnrichedListing2Vec(nn.Module):
    """
    Listing2Vec with feature-enriched embeddings.
    
    12-year-old version:
    Instead of just giving each listing a secret code based on who clicked it,
    we also add extra information: what type of place it is, how much it costs,
    and where it is located. This gives the model a head start.
    
    Staff-level detail:
    - Behavioral embedding: learned from session co-occurrence (Skip-gram)
    - Feature embeddings: separate embedding tables for categorical features
    - Final embedding = projection of concatenated behavioral + feature embeddings
    - Helps cold-start: even without behavioral data, features provide a reasonable embedding
    """
    def __init__(self, num_listings, num_markets, num_price_buckets, 
                 behavioral_dim=24, feature_dim=8, output_dim=32):
        super().__init__()
        
        # Behavioral embeddings (learned from sessions)
        self.behavioral_emb = nn.Embedding(num_listings, behavioral_dim)
        
        # Feature embeddings
        self.market_emb = nn.Embedding(num_markets, feature_dim)
        self.price_emb = nn.Embedding(num_price_buckets, feature_dim)
        
        # Projection layer: combine all into a single embedding
        combined_dim = behavioral_dim + feature_dim * 2
        self.projection = nn.Linear(combined_dim, output_dim)
        
        # Initialize
        nn.init.xavier_uniform_(self.behavioral_emb.weight)
        nn.init.xavier_uniform_(self.market_emb.weight)
        nn.init.xavier_uniform_(self.price_emb.weight)
    
    def get_combined_embedding(self, listing_ids, market_ids, price_ids):
        """Get the full enriched embedding."""
        beh = self.behavioral_emb(listing_ids)
        mkt = self.market_emb(market_ids)
        prc = self.price_emb(price_ids)
        
        combined = torch.cat([beh, mkt, prc], dim=-1)
        return self.projection(combined)
    
    def forward(self, center_ids, center_markets, center_prices,
                context_ids, context_markets, context_prices,
                negative_ids, negative_markets, negative_prices):
        
        center_emb = self.get_combined_embedding(center_ids, center_markets, center_prices)
        context_emb = self.get_combined_embedding(context_ids, context_markets, context_prices)
        
        # Positive score
        pos_score = (center_emb * context_emb).sum(dim=1)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-10)
        
        # Negative scores
        neg_emb = self.get_combined_embedding(negative_ids.view(-1), 
                                              negative_markets.view(-1),
                                              negative_prices.view(-1))
        neg_emb = neg_emb.view(negative_ids.shape[0], negative_ids.shape[1], -1)
        neg_score = torch.bmm(neg_emb, center_emb.unsqueeze(2)).squeeze(2)
        neg_loss = -torch.log(torch.sigmoid(-neg_score) + 1e-10).sum(dim=1)
        
        return (pos_loss + neg_loss).mean()


# Create the model
market_names = list(markets.keys())
market_to_idx = {name: idx for idx, name in enumerate(market_names)}

enriched_model = FeatureEnrichedListing2Vec(
    num_listings=NUM_LISTINGS,
    num_markets=len(markets),
    num_price_buckets=5,
    behavioral_dim=24,
    feature_dim=8,
    output_dim=32
)

print("Feature-Enriched Listing2Vec:")
print(f"  Behavioral embedding dim: 24")
print(f"  Market embedding dim:     8")
print(f"  Price embedding dim:      8")
print(f"  Combined before projection: 40")
print(f"  Final output dim:         32")
print(f"\nTotal parameters: {sum(p.numel() for p in enriched_model.parameters())}")
print(f"\nAdvantage: Even a NEW listing with no clicks gets a reasonable embedding")
print(f"from its market and price features alone.")

## Key Takeaways

1. **Word2Vec is the foundation**: listings = words, sessions = sentences, similarity comes from co-occurrence
2. **Skip-gram > CBOW** for listings because many listings are rare (few interactions)
3. **Negative sampling** reduces training from O(V) to O(k) per step -- essential for 5M listings
4. **Hard negatives** (same-market) force fine-grained intra-market discrimination -- the most impactful improvement
5. **Global context** (booked listing) aligns embeddings with the business objective (bookings, not just clicks)
6. **Feature-enriched embeddings** help with cold start by providing a reasonable initial embedding from listing attributes
7. **Cold-start strategies**: geographic proxy, feature-based initialization, and daily retraining
8. **Daily retraining** ensures new listings get their own learned embeddings within 24 hours

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)